# Proyek Analisis Data: Bike Sharing Dataset
- **Nama:** Arnan Refael Adriwinoto
- **Email:** alicizations03@gmail.com
- **ID Dicoding:** rvolcy

## Menentukan Pertanyaan Bisnis

- Faktor apa yang paling memengaruhi jumlah penyewaan sepeda per hari?
- Bagaimana pengaruh musim dan kondisi cuaca terhadap jumlah penyewaan sepeda?

## Import Semua Packages/Library yang Digunakan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st


## Data Wrangling

### Gathering Data

In [ ]:
# Import library yang dibutuhkan
import pandas as pd

# Membaca dataset harian (day.csv)
df = pd.read_csv('data/day.csv')

# Menampilkan 5 baris pertama
df.head()

**Insight:**
- Dataset ini berisi data penyewaan sepeda harian dari sistem Bike Sharing selama tahun 2011–2012.
- Terdapat beberapa kolom seperti season, yr, mnth, holiday, weekday, temp, humidity, windspeed, casual, registered, dan cnt (total sewa).
- Dataset masih memiliki nama kolom yang belum ramah dibaca, seperti yr dan mnth, sehingga perlu diubah nanti di tahap cleaning.

### Assessing Data

In [ ]:
# Mengecek informasi umum dataset
df.info()

# Mengecek apakah ada nilai kosong
print("\nJumlah nilai kosong per kolom:")
print(df.isnull().sum())

# Mengecek data duplikat
print("\nJumlah data duplikat:", df.duplicated().sum())

# Melihat statistik deskriptif
df.describe()

**Insight:**
- Tidak ditemukan nilai kosong (NaN) dalam dataset, artinya data sudah lengkap.
- Tidak ada data duplikat yang berarti dataset bersih secara struktur.
- Kolom temp, atemp, humidity, dan windspeed semuanya berupa angka desimal yang sudah dinormalisasi (0–1).
- Kolom casual, registered, dan cnt (jumlah total peminjaman) menunjukkan distribusi nilai yang cukup lebar, menandakan variasi pemakaian sepeda yang tinggi antar hari.
- Perlu dilakukan renaming agar kolom lebih mudah dipahami pada tahap berikutnya.

### Cleaning Data

In [ ]:
import pandas as pd

# Baca ulang data mentah
df = pd.read_csv('data/day.csv')

# Ubah nama kolom agar lebih mudah dipahami
df = df.rename(columns={
    'yr': 'year',
    'mnth': 'month',
    'weathersit': 'weather',
    'hum': 'humidity',
    'cnt': 'count'
})

# Hapus data duplikat dan nilai kosong
df = df.drop_duplicates()
df = df.dropna()

# Simpan hasil data bersih ke folder dashboard
df.to_csv('dashboard/main_data.csv', index=False)

# Tampilkan hasil akhir untuk konfirmasi
df.head()

**Insight:**
- Nama kolom diubah supaya lebih deskriptif dan mudah dibaca saat analisis (yr → year, mnth → month, hum → humidity, dll).
- Tidak ditemukan nilai NaN ataupun duplikat, tapi kode tetap disertakan untuk berjaga-jaga kalau dataset diperbarui.
- Setelah dibersihkan, data disimpan ke file dashboard/main_data.csv, yang nantinya akan digunakan Streamlit sebagai sumber data utama.
- Hasil cleaning membuat dataset lebih konsisten dan siap untuk tahap Exploratory Data Analysis (EDA) selanjutnya.

## Exploratory Data Analysis (EDA)

### Explore ...

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ukuran figure default
plt.rcParams['figure.figsize'] = (10, 5)
sns.set(style="whitegrid")

# Statistik deskriptif
print(df.describe())

# Distribusi jumlah penyewaan sepeda per hari
sns.histplot(df['count'], bins=30, kde=True)
plt.title("Distribusi Jumlah Penyewaan Sepeda per Hari")
plt.xlabel("Jumlah Penyewaan")
plt.ylabel("Frekuensi")
plt.show()

**Insight:**
- Dari distribusi data, terlihat bahwa sebagian besar jumlah penyewaan sepeda berada pada rentang 500–4500 per hari.
- Terdapat sedikit data ekstrem di atas 6000, yang kemungkinan terjadi pada musim panas atau akhir pekan dengan cuaca cerah.
- Secara umum, data tampak positif skewed, menunjukkan bahwa sebagian besar hari memiliki jumlah penyewaan yang tergolong sedang.

## Visualization & Explanatory Analysis

### Pertanyaan 1:

In [ ]:
# Korelasi antar variabel numerik
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Korelasi Antar Variabel")
plt.show()

### Pertanyaan 2:

In [ ]:
# Rata-rata sewa berdasarkan musim
season_avg = df.groupby('season')['count'].mean().reset_index()

plt.figure(figsize=(7,5))
sns.barplot(x='season', y='count', hue='season', data=season_avg, palette='viridis', legend=False)
plt.title('Rata-rata Penyewaan per Musim')
plt.xlabel('Musim')
plt.ylabel('Rata-rata Jumlah Sewa')
plt.show()

# Rata-rata sewa berdasarkan cuaca
weather_avg = df.groupby('weather')['count'].mean().reset_index()

plt.figure(figsize=(7,5))
sns.barplot(x='weather', y='count', hue='weather', data=weather_avg, palette='cool', legend=False)
plt.title('Rata-rata Penyewaan per Kondisi Cuaca')
plt.xlabel('Kondisi Cuaca')
plt.ylabel('Rata-rata Jumlah Sewa')
plt.show()

**Insight:**
- Faktor paling berpengaruh terhadap jumlah penyewaan sepeda: suhu (positif) dan kelembapan (negatif).
- Musim dan cuaca: penyewaan meningkat pada musim panas dengan cuaca cerah, dan menurun drastis pada musim dingin atau saat hujan.
- Secara umum, kenyamanan cuaca adalah indikator utama dalam tren penyewaan sepeda harian.

## Analisis Lanjutan (Opsional)

In [ ]:
# Pastikan kolom tanggal dalam format datetime
df['dteday'] = pd.to_datetime(df['dteday'])

# Definisikan tanggal referensi (misal tanggal terakhir di dataset)
reference_date = df['dteday'].max()

# Hitung RFM
rfm = df.groupby('dteday').agg({
    'count': ['sum', 'mean']
}).reset_index()

rfm.columns = ['dteday', 'Monetary', 'Frequency']
rfm['Recency'] = (reference_date - rfm['dteday']).dt.days

print(rfm.head())

## Conclusion

- Faktor paling berpengaruh terhadap penyewaan sepeda adalah temperatur. Semakin hangat, semakin tinggi jumlah penyewaan.
- Musim panas dan cuaca cerah memicu peningkatan signifikan dalam jumlah penyewa.
- Berdasarkan RFM analysis, tren penyewaan meningkat mendekati akhir periode (recency rendah), terutama di akhir pekan dan musim hangat.
- Strategi bisnis: perbanyak unit dan promo saat musim panas & hari cerah, serta maintenance sepeda di musim hujan karena penurunan aktivitas.